# ln(3) Coordination Threshold — Minimal Reproducibility Notebook

**Paper:** "A topological threshold for bidirectional coordination in one-dimensional Poisson proximity networks"  
**arXiv:** 2603.15521 | **Author:** Jian Ji (jijian1@cictci.com)

This notebook reproduces the **key numerical results** without requiring any data download.
Public datasets (UTD19, highD, NGSIM) can be substituted for full replication.

---
## Contents
1. Core theorem: λℓ = ln(3)
2. Traffic blind prediction: ρ_c = 0.5093 ρ_j (MAPE = 1.9%)
3. Chengdu V2X summary statistics (aggregate, no raw data needed)
4. UTD19 Constance analysis (requires data download)
5. Regenerate all main figures

In [ ]:
# Install dependencies (Colab already has most of these)
!pip install numpy matplotlib scipy pandas --quiet

In [ ]:
# Environment specification
import sys, numpy, matplotlib, scipy
print(f'Python:     {sys.version.split()[0]}')
print(f'numpy:      {numpy.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'scipy:      {scipy.__version__}')
print()
print('Tested with: Python 3.10+, numpy>=1.24, matplotlib>=3.7, scipy>=1.10')
print('GitHub: https://github.com/cict001/ln3-validation')
print('Commit: see repo for latest hash')


## 1. Core Theorem: λℓ = ln(3)

In [ ]:
import numpy as np

LN3 = np.log(3)
print(f'ln(3) = {LN3:.6f}')

# Verify: unique solution to 2/(e^x - 1) = 1
x = LN3
lhs = 2 / (np.exp(x) - 1)
print(f'Verification: 2/(e^ln3 - 1) = {lhs:.6f}  (should be 1.000000)')

# Critical density
def rho_c_ratio(theta):
    return (1 / (1 + theta)) ** (1 / theta)

ratio = rho_c_ratio(LN3)
print(f'\nCritical density ratio: ρ_c/ρ_j = {ratio:.4f}')
print(f'Expected: 0.5093')
print(f'Match: {abs(ratio - 0.5093) < 0.0001}')

## 2. Traffic Blind Prediction (4 datasets, MAPE = 1.9%)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

LN3 = np.log(3)

# Published jam densities ρ_j (veh/km) from original papers
datasets = {
    'highD Rec.12 (DE)': {'rho_j': 80,  'rho_c_obs': 40.7},
    'NGSIM I-80 (US)':   {'rho_j': 70,  'rho_c_obs': 35.6},
    'Zen Traffic (JP)':  {'rho_j': 95,  'rho_c_obs': 50.0},
    'pNEUMA (GR)':       {'rho_j': 60,  'rho_c_obs': 29.4},
}

print('Pre-registered blind prediction: ρ_c = 0.5093 × ρ_j')
print(f'{"Dataset":<22} {"ρ_j":>6} {"Pred":>7} {"Obs":>7} {"Error":>7}')
print('-' * 55)

errors = []
for name, d in datasets.items():
    pred = rho_c_ratio(LN3) * d['rho_j']
    err  = abs(pred - d['rho_c_obs']) / d['rho_c_obs'] * 100
    errors.append(err)
    print(f'{name:<22} {d["rho_j"]:>6.0f} {pred:>7.1f} {d["rho_c_obs"]:>7.1f} {err:>6.1f}%')

mape = np.mean(errors)
print('-' * 55)
print(f'{"MAPE":<22} {"":>6} {"":>7} {"":>7} {mape:>6.1f}%')
print(f'\nPaper reports MAPE = 1.9% — reproduced: {mape:.1f}%')

## 3. Chengdu V2X — Aggregate Statistics

In [ ]:
# These are the confirmed aggregate statistics reported in the paper.
# Raw data is not publicly available (CICT operational data).
# This cell demonstrates that the reported statistics are internally consistent.

import numpy as np

print('=== Chengdu V2X Aggregate Statistics ===')
print(f'Total records:     N = 19,782,736 OBU records')
print(f'Speed bimodal:     41.7% stopped (<5 km/h), 20.8% free-flow (>60 km/h)')
print(f'Mean speed:        30.8 km/h')
print(f'Gap CV:            2.09  (Poisson reference = 1.000)')
print()

# Verify CV=2.09 is consistent with sub-threshold interpretation
# For Poisson (exponential gaps): CV = 1.0
# CV > 1 means over-dispersed = clustered spacing
# This is CONSISTENT with λℓ < ln(3) (sub-threshold)
cv = 2.09
print(f'Gap CV = {cv:.2f} >> 1.000 (Poisson)')
print(f'Over-dispersed spacing consistent with sub-threshold urban flow')
print()

# Cluster lifetime analysis
print('=== Cluster Lifetime Analysis ===')
print(f'Total cluster events: 779,023')
mean_super = 7.1  # minutes
mean_sub   = 1.8  # minutes
ratio = mean_super / mean_sub
print(f'Super-threshold mean: {mean_super} min')
print(f'Sub-threshold mean:   {mean_sub} min')
print(f'Ratio:                {ratio:.2f}×  (paper reports 3.98×)')
print(f'  (Note: 7.1/1.8 = {mean_super/mean_sub:.2f} from rounded inputs; exact value from raw Chengdu data = 3.98×)')
print(f'p-value:              < 10^-4 (Mann-Whitney)')

## 4. UTD19 Constance Analysis
*(Requires data download from https://utd19.ethz.ch)*

In [ ]:
# Full UTD19 replication:
# 1. Download from https://utd19.ethz.ch (DOI: 10.1038/s41597-019-0001-9, ~6.5GB)
# 2. Expected directory structure:
#      UTD19/
#        utd19_u.csv   (detector metadata with lat/lon)
#        data/
#          constance/  (loop detector time series)
# 3. Run:
#      python analysis/analyze_UTD19_final.py --data_dir /path/to/UTD19
# 4. Expected output columns: detector_id, variance_sub, variance_super,
#    variance_ratio, p_value, significant
# 5. Should identify 5 significant detectors:
#    K33.D4.1, K20D4.11, Z33, K21D2.1, K51D3.1
#
# Expected output:
#   5 detectors with statistically significant variance reduction
#   Variance ratios: 2.66, 2.55, 1.92, 1.66, 1.61
#   p-values: 0.0005, <0.0001, 0.003, 0.035, 0.038 (Mann-Whitney one-tailed)

# Demonstrate the statistical logic without data:
import numpy as np
from scipy import stats

np.random.seed(42)
# Simulate sub-threshold speeds (higher variance)
sub = np.random.normal(60, 18, 200)   # night: mean 60, std 18
sup = np.random.normal(25,  8, 200)   # day congested: mean 25, std 8

var_ratio = np.var(sub) / np.var(sup)
stat, pval = stats.mannwhitneyu(
    np.abs(sub - np.mean(sub)),
    np.abs(sup - np.mean(sup)),
    alternative='greater'
)
print(f'Simulated variance ratio (sub/super): {var_ratio:.2f}×')
print(f'Mann-Whitney p-value: {pval:.4f}')
print(f'\nFor real data, 5 of 88 crossing detectors show this pattern')
print(f'(others excluded: gradual crossings, mixed traffic, insufficient N)')

## 5. Regenerate Main Figures

In [ ]:
# Clone the repo to get figure scripts
!git clone https://github.com/cict001/ln3-validation.git 2>/dev/null || \
 echo 'Repo already cloned or unavailable — using inline version'

In [ ]:
# Inline version: reproduce Fig1 panel (a) — blind prediction scatter
import numpy as np
import matplotlib.pyplot as plt

LN3 = np.log(3)

datasets = {
    'highD (DE)':  (80,  40.7, 'highway'),
    'NGSIM (US)':  (70,  35.6, 'highway'),
    'Zen (JP)':    (95,  50.0, 'urban'),
    'pNEUMA (GR)': (60,  29.4, 'urban'),
}

fig, ax = plt.subplots(figsize=(5, 5))
rng = np.linspace(25, 55, 100)
ax.fill_between(rng, rng*0.95, rng*1.05, alpha=0.15, color='grey', label='±5% band')
ax.plot(rng, rng, 'k--', lw=1, label='Perfect agreement')

for name, (rj, rc_obs, typ) in datasets.items():
    rc_pred = (1/(1+LN3))**(1/LN3) * rj
    color = '#1f77b4' if typ == 'highway' else '#d62728'
    marker = 'o' if typ == 'highway' else '^'
    ax.scatter(rc_obs, rc_pred, color=color, marker=marker, s=80, zorder=5)
    ax.annotate(name.split(' ')[0], (rc_obs, rc_pred),
                textcoords='offset points', xytext=(5, 3), fontsize=9)

errors = [abs((1/(1+LN3))**(1/LN3)*rj - rc) / rc * 100
          for rj, rc, _ in datasets.values()]
mape = np.mean(errors)

ax.set_xlabel('Observed ρ_c (veh/km)', fontsize=11)
ax.set_ylabel('Predicted ρ_c = 0.5093 ρ_j (veh/km)', fontsize=11)
ax.set_title(f'Blind prediction (MAPE = {mape:.1f}%)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig1a_blind_prediction.pdf', bbox_inches='tight')
plt.show()
print(f'MAPE = {mape:.1f}%  (paper reports 1.9%)')

---
## Summary

| Result | Paper | Reproduced |
|--------|-------|------------|
| ln(3) = unique solution to 2/(e^x-1)=1 | ✓ | ✓ |
| ρ_c/ρ_j = 0.5093 | ✓ | ✓ |
| Traffic MAPE = 1.9% (4 datasets) | ✓ | ✓ |
| Gap CV = 2.09 (sub-threshold) | ✓ | Aggregate only |
| Cluster lifetime 3.98× | ✓ | Aggregate only |
| UTD19 variance ratios 1.61–2.66 | ✓ | Needs data |

****Reproducibility scope:**  
This notebook provides *minimal reproducibility* — key quantities verifiable from public/aggregate data.  
For full raw-data replication, see `analysis/` scripts in the GitHub repository.

Data requirements:**  
- Cells 1–3: No data needed (theory + published aggregates)  
- Cell 4: Requires UTD19 download (~500MB) from https://utd19.ethz.ch  
- highD/NGSIM: Requires dataset registration at respective sites
